In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go

colors = {
    "Walk": "#e87ba4",
    "Bike": "#eda100",
    "Bus": "#1baf7a",
    "Train": "#eb6834",
    "Carpool": "#008300",
    "Car": "#2a78d6",
}

min_count = 5
BIN = 5  # width of each time window in minutes

root = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent

df = pd.read_csv(root / "data" / "processed" / "commutes_clean.csv")

bins = np.arange(0, df["duration_minutes"].max() + BIN, BIN)
starts, ends = bins[:-1], bins[1:]
centers = starts + BIN / 2
windows = np.array([f"{int(s)}-{int(e)}" for s, e in zip(starts, ends)])  # "70-75", shown in the pop-up

fig = go.Figure()

for mode, color in colors.items():
    minutes = df.loc[df["mode_of_transportation"] == mode, "duration_minutes"]
    n = len(minutes)
    if n == 0:
        continue

    counts, _ = np.histogram(minutes, bins=bins)

    if n >= min_count:
        keep = np.ones(len(counts), dtype=bool)   # draw every window as a line
        draw_mode, label, size = "lines+markers", f"{mode} (n={n})", 6
    else:
        keep = counts > 0                         # dots only where there are commutes
        draw_mode, label, size = "markers", f"{mode} (n={n}, dots only)", 10

    fig.add_trace(go.Scatter(
        x=centers[keep],
        y=counts[keep],
        mode=draw_mode,
        name=label,
        line=dict(color=color, width=2),
        marker=dict(color=color, size=size),
        customdata=windows[keep],
        hovertemplate=(
            f"<b>{mode}</b><br>"
            "Commute time: %{customdata} min<br>"
            "Commutes: %{y}"
            "<extra></extra>"
        ),
    ))

fig.update_layout(
    title="Commute duration by mode of transportation",
    xaxis_title="Commute time (minutes, 5-minute windows)",
    yaxis_title="Number of commutes",
    template="plotly_white",
    hovermode="closest",
    width=1000,
    height=600,
)
fig.update_xaxes(rangemode="tozero")
fig.update_yaxes(rangemode="tozero")
fig.show()